# Tool Use & Function Calling — Teaching LLMs to Use Tools

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/03_tool_use_and_function_calling.ipynb)

## What This Notebook Teaches You

LLMs can only **generate text**. They cannot calculate, look up real-time data, or interact with systems. **Tools** bridge this gap — they let an LLM decide *what* to do, while code decides *how* to do it.

By the end of this notebook, you will be able to:

1. **Explain why tools matter** — the fundamental limitation of LLMs and how tools solve it
2. **Build tool schemas** — define tools with typed parameters using dataclasses
3. **Create a ToolRegistry** — automatically extract schemas from Python functions using `inspect`
4. **Write a ToolDispatcher** — parse LLM output, validate calls, execute tools, handle errors
5. **Build a ToolAgent** — integrate tools with the agent loop from Notebook 02
6. **Diagnose failure modes** — hallucinated tools, wrong params, malformed JSON

---

> **Prerequisites**: Complete Notebook 02 (The Agent Loop).
> **Runtime**: Google Colab with T4 GPU (free tier).
> **Time**: ~45–60 minutes.

## Part 0: Environment Setup

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Tools Matter

### 1.1 — The Fundamental Limitation

LLMs are trained on text. They can *describe* math, but they can't *do* math. They can *talk about* today's date, but they don't *know* today's date. This creates a reliability gap:

| Task | LLM Alone | LLM + Tool |
|------|-----------|------------|
| "What is 347 × 829?" | Often wrong (guesses) | Always correct (calculator) |
| "What day is today?" | Hallucinated date | Accurate (system clock) |
| "Count the words in this text" | Approximate | Exact (code) |
| "Sort these numbers" | May miss/swap items | Perfect (algorithm) |

### 1.2 — The Research Foundation

**Schick et al. (2023), "Toolformer"** showed that LLMs can learn to *decide when and how* to use tools via a simple protocol:
- The model generates a special token indicating a tool call
- An external system executes the tool
- The result is inserted back into the context

This insight powers modern function calling in GPT-4, Claude, Gemini, and open-source models.

### 1.3 — How Tool Use Works

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│              │     │              │     │              │
│  User Task   │────►│  LLM Decides │────►│  Tool Executes│
│              │     │  Which Tool  │     │  Returns Result│
│              │     │  + Arguments │     │              │
└──────────────┘     └──────────────┘     └───────┬──────┘
                                                   │
                     ┌──────────────┐              │
                     │  LLM Forms   │◄─────────────┘
                     │  Final Answer│
                     └──────────────┘
```

The key insight: the LLM is the **decision maker** (which tool, what arguments), but the **code is the executor** (reliable, deterministic).

## 2. Tool Schemas — Defining What Tools Look Like

### 2.1 — Why Schemas Matter

For an LLM to use a tool, it needs to know:
1. **Name** — what to call it
2. **Description** — what it does (in natural language)
3. **Parameters** — what inputs it takes, with types and descriptions
4. **Return type** — what it gives back

We define this as structured data using dataclasses.

In [ ]:
@dataclass
class ParameterSchema:
    """Schema for a single tool parameter."""
    name: str
    type: str  # "string", "int", "float", "bool", "list"
    description: str
    required: bool = True
    default: Any = None

@dataclass
class Tool:
    """Complete tool definition with schema and implementation."""
    name: str
    description: str
    parameters: List[ParameterSchema]
    function: Callable
    return_type: str = "string"

    def to_prompt_string(self) -> str:
        """Format this tool for inclusion in a system prompt."""
        params_str = ", ".join(
            f"{p.name}: {p.type}" + ("" if p.required else f" = {p.default}")
            for p in self.parameters
        )
        param_details = "\n".join(
            f"    - {p.name} ({p.type}{'required' if p.required else ', optional'}): {p.description}"
            for p in self.parameters
        )
        return (
            f"  {self.name}({params_str}) -> {self.return_type}\n"
            f"    Description: {self.description}\n"
            f"    Parameters:\n{param_details}"
        )

# Example
example_param = ParameterSchema(name="expression", type="string", description="Math expression to evaluate")
example_tool = Tool(
    name="calculator",
    description="Evaluates a mathematical expression",
    parameters=[example_param],
    function=lambda expression: str(eval(expression)),
    return_type="string",
)
print("Example tool schema:")
print(example_tool.to_prompt_string())

## 3. Building a ToolRegistry

### 3.1 — Automatic Schema Extraction

Manually defining `ParameterSchema` for every tool is tedious. We can use Python's `inspect` module and `get_type_hints()` to **automatically** extract schemas from function signatures.

In [ ]:
# Type mapping for inspect-based extraction
TYPE_MAP = {
    str: "string",
    int: "int",
    float: "float",
    bool: "bool",
    list: "list",
    dict: "dict",
    "str": "string",
    "int": "int",
    "float": "float",
    "bool": "bool",
    "list": "list",
    "dict": "dict",
}

class ToolRegistry:
    """Registry that auto-extracts tool schemas from Python functions."""

    def __init__(self):
        self.tools: Dict[str, Tool] = {}

    def register_tool(self, func: Callable = None, *, name: str = None,
                      description: str = None) -> Callable:
        """Register a function as a tool. Can be used as a decorator."""
        def decorator(f):
            tool_name = name or f.__name__
            tool_desc = description or (f.__doc__ or "No description available").strip().split("\n")[0]

            # Extract parameters from function signature
            sig = inspect.signature(f)
            hints = get_type_hints(f) if hasattr(f, '__annotations__') else {}

            parameters = []
            for param_name, param in sig.parameters.items():
                param_type = hints.get(param_name, str)
                type_str = TYPE_MAP.get(param_type, "string")
                has_default = param.default is not inspect.Parameter.empty
                parameters.append(ParameterSchema(
                    name=param_name,
                    type=type_str,
                    description=f"Parameter: {param_name}",
                    required=not has_default,
                    default=param.default if has_default else None,
                ))

            # Determine return type
            return_hint = hints.get("return", str)
            return_type = TYPE_MAP.get(return_hint, "string")

            tool = Tool(
                name=tool_name,
                description=tool_desc,
                parameters=parameters,
                function=f,
                return_type=return_type,
            )
            self.tools[tool_name] = tool
            return f

        if func is not None:
            return decorator(func)
        return decorator

    def get_tool(self, name: str) -> Optional[Tool]:
        """Get a tool by name."""
        return self.tools.get(name)

    def list_tools(self) -> List[str]:
        """List all registered tool names."""
        return list(self.tools.keys())

    def format_for_prompt(self) -> str:
        """Format all tools for inclusion in the system prompt."""
        lines = ["Available tools:"]
        for tool in self.tools.values():
            lines.append(tool.to_prompt_string())
        return "\n\n".join(lines)

print("✓ ToolRegistry class defined")

## 4. Building Example Tools

We create 8 diverse tools to demonstrate different types of capabilities.

In [ ]:
registry = ToolRegistry()

@registry.register_tool(description="Evaluate a mathematical expression safely")
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return f"Error: invalid characters in expression: {expression}"
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

@registry.register_tool(description="Count the number of characters in a text string")
def string_length(text: str) -> str:
    """Return the length of a string."""
    return str(len(text))

@registry.register_tool(description="Count the number of words in a text string")
def word_count(text: str) -> str:
    """Count words in text."""
    words = text.split()
    return str(len(words))

@registry.register_tool(description="Get the current date and time")
def get_date() -> str:
    """Return current date and time."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@registry.register_tool(description="Reverse a text string")
def reverse_string(text: str) -> str:
    """Reverse the given string."""
    return text[::-1]

@registry.register_tool(description="Sort a comma-separated list of items")
def list_sort(items: str) -> str:
    """Sort a comma-separated list. Tries numeric sort first, falls back to alphabetic."""
    parts = [x.strip() for x in items.split(",")]
    try:
        sorted_parts = sorted(parts, key=float)
    except ValueError:
        sorted_parts = sorted(parts)
    return ", ".join(sorted_parts)

# Built-in dictionary for lookup
DICTIONARY = {
    "algorithm": "A step-by-step procedure for solving a problem or accomplishing a task",
    "binary": "A number system using only 0 and 1, the basis of digital computing",
    "cache": "A hardware or software component that stores data for faster future access",
    "debugging": "The process of finding and fixing errors in computer programs",
    "encryption": "The process of converting data into a coded format to prevent unauthorized access",
    "firewall": "A network security system that monitors and controls incoming and outgoing traffic",
    "git": "A distributed version control system for tracking changes in source code",
    "hash": "A function that maps data of arbitrary size to fixed-size values",
    "iteration": "The repetition of a process or set of instructions in a program",
    "json": "JavaScript Object Notation, a lightweight data interchange format",
    "kernel": "The core component of an operating system that manages system resources",
    "latency": "The time delay between a request and the corresponding response",
    "middleware": "Software that acts as a bridge between an operating system and applications",
    "node": "A basic unit of a data structure, such as a linked list or tree",
    "object": "A self-contained entity with properties and methods in object-oriented programming",
    "protocol": "A set of rules governing the format and transmission of data",
    "query": "A request for data or information from a database",
    "recursion": "A technique where a function calls itself to solve smaller instances of a problem",
    "syntax": "The set of rules that defines the structure of a programming language",
    "thread": "The smallest unit of execution within a process",
}

@registry.register_tool(description="Look up the definition of a computing/programming term")
def dictionary_lookup(term: str) -> str:
    """Look up a term in the computing dictionary."""
    term_lower = term.lower().strip()
    if term_lower in DICTIONARY:
        return f"{term}: {DICTIONARY[term_lower]}"
    # Fuzzy match
    matches = [k for k in DICTIONARY if term_lower in k or k in term_lower]
    if matches:
        return f"Did you mean: {', '.join(matches)}? Definition of {matches[0]}: {DICTIONARY[matches[0]]}"
    return f"Term '{term}' not found. Available: {', '.join(sorted(DICTIONARY.keys()))}"

@registry.register_tool(description="Analyze text and return statistics: character count, word count, sentence count, and average word length")
def text_statistics(text: str) -> str:
    """Return comprehensive statistics about a text."""
    chars = len(text)
    words = text.split()
    word_count_val = len(words)
    sentences = len(re.split(r'[.!?]+', text.strip())) - (1 if text.strip()[-1:] in '.!?' else 0)
    sentences = max(sentences, 1)
    avg_word_len = sum(len(w.strip(".,!?;:")) for w in words) / max(word_count_val, 1)
    return (
        f"Characters: {chars}, Words: {word_count_val}, "
        f"Sentences: {sentences}, Avg word length: {avg_word_len:.1f}"
    )

# Display all tools
print(f"Registered {len(registry.tools)} tools:\n")
print(registry.format_for_prompt())

## 5. The Tool-Use System Prompt

The system prompt must tell the LLM:
1. What tools are available (with schemas)
2. How to call them (JSON format)
3. How to provide a final answer
4. Rules and constraints

In [ ]:
def build_tool_system_prompt(registry: ToolRegistry) -> str:
    """Build a system prompt that includes tool descriptions."""
    tools_desc = registry.format_for_prompt()
    return f"""You are a helpful assistant with access to tools. Use tools when you need to perform calculations, look up information, or manipulate data.

{tools_desc}

On each turn, respond with EXACTLY one JSON object (no other text):

To use a tool:
{{"action": "tool", "tool": "<tool_name>", "arguments": {{"param1": "value1", "param2": "value2"}}}}

To provide a final answer (after you have all information needed):
{{"action": "answer", "answer": "your complete final answer"}}

To think/reason (when you need to plan before acting):
{{"action": "think", "thought": "your reasoning here"}}

Rules:
- Use tools for calculations, data lookup, and text manipulation — do NOT guess.
- After receiving a tool result, use it to inform your answer.
- Always respond with ONLY a JSON object.
- Provide your final answer only when you have all the information you need.
"""

prompt = build_tool_system_prompt(registry)
print("System prompt preview (first 500 chars):")
print(prompt[:500])
print(f"\n... ({len(prompt)} chars total)")

## 6. Building the ToolDispatcher

The dispatcher bridges the LLM's decisions and actual tool execution. It must:
1. Parse the LLM's JSON output
2. Validate the tool name exists
3. Validate the arguments match the schema
4. Execute the tool safely
5. Format the result for the LLM
6. Handle all errors gracefully

In [ ]:
class ToolDispatcher:
    """Dispatches tool calls from LLM output to actual tool functions."""

    def __init__(self, registry: ToolRegistry):
        self.registry = registry
        self.call_history: List[Dict[str, Any]] = []

    def parse_llm_output(self, text: str) -> Dict[str, Any]:
        """Parse LLM output into a structured action."""
        text = text.strip()

        # Try direct JSON parse
        try:
            result = json.loads(text)
            if "action" in result:
                return result
        except json.JSONDecodeError:
            pass

        # Try extracting JSON from code blocks
        code_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
        if code_match:
            try:
                result = json.loads(code_match.group(1))
                if "action" in result:
                    return result
            except json.JSONDecodeError:
                pass

        # Try finding JSON object in text
        brace_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text)
        if brace_match:
            try:
                result = json.loads(brace_match.group())
                if "action" in result:
                    return result
            except json.JSONDecodeError:
                pass

        return {"action": "think", "thought": text[:500]}

    def validate_tool_call(self, parsed: Dict[str, Any]) -> Tuple[bool, str]:
        """Validate a tool call: tool exists and arguments are reasonable."""
        if parsed.get("action") != "tool":
            return True, "Not a tool call"

        tool_name = parsed.get("tool", "")
        tool = self.registry.get_tool(tool_name)
        if not tool:
            available = ", ".join(self.registry.list_tools())
            return False, f"Unknown tool '{tool_name}'. Available: {available}"

        arguments = parsed.get("arguments", {})
        if not isinstance(arguments, dict):
            return False, f"Arguments must be a dict, got {type(arguments).__name__}"

        # Check required parameters
        for param in tool.parameters:
            if param.required and param.name not in arguments:
                return False, f"Missing required parameter '{param.name}' for tool '{tool_name}'"

        return True, "Valid"

    def execute(self, parsed: Dict[str, Any]) -> str:
        """Execute a tool call and return the result as a string."""
        tool_name = parsed.get("tool", "")
        tool = self.registry.get_tool(tool_name)
        arguments = parsed.get("arguments", {})

        call_record = {
            "tool": tool_name,
            "arguments": arguments,
            "success": False,
            "result": None,
            "error": None,
        }

        try:
            # Filter arguments to only accepted parameters
            sig = inspect.signature(tool.function)
            valid_args = {k: v for k, v in arguments.items() if k in sig.parameters}
            result = tool.function(**valid_args)
            call_record["success"] = True
            call_record["result"] = str(result)
        except Exception as e:
            call_record["error"] = str(e)
            call_record["result"] = f"Error executing {tool_name}: {str(e)}"

        self.call_history.append(call_record)
        return call_record["result"]

    def dispatch(self, text: str) -> Tuple[Dict[str, Any], Optional[str]]:
        """Full dispatch pipeline: parse → validate → execute."""
        parsed = self.parse_llm_output(text)

        if parsed.get("action") != "tool":
            return parsed, None

        valid, msg = self.validate_tool_call(parsed)
        if not valid:
            return parsed, f"Tool call error: {msg}"

        result = self.execute(parsed)
        return parsed, result

# Test the dispatcher
dispatcher = ToolDispatcher(registry)

test_outputs = [
    '{"action": "tool", "tool": "calculator", "arguments": {"expression": "347 * 829"}}',
    '{"action": "tool", "tool": "string_length", "arguments": {"text": "Hello, World!"}}',
    '{"action": "tool", "tool": "nonexistent", "arguments": {}}',
    '{"action": "answer", "answer": "The result is 42"}',
]

for output in test_outputs:
    parsed, result = dispatcher.dispatch(output)
    print(f"Input:  {output[:70]}...")
    print(f"Parsed: {parsed}")
    if result is not None:
        print(f"Result: {result}")
    print()

## 7. Building the ToolAgent

Now we integrate everything: registry + dispatcher + agent loop. The agent can think, use tools, and answer.

In [ ]:
class ToolAgent:
    """Agent that can use tools to solve tasks."""

    def __init__(self, registry: ToolRegistry, max_steps: int = 10):
        self.registry = registry
        self.dispatcher = ToolDispatcher(registry)
        self.max_steps = max_steps
        self.system_prompt = build_tool_system_prompt(registry)

    def run(self, task: str) -> Dict[str, Any]:
        """Run the agent on a task."""
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Task: {task}"},
        ]

        trace = []
        final_answer = None
        start_time = time.time()

        print(f"{'='*60}")
        print(f"ToolAgent: {task}")
        print(f"{'='*60}")

        for step in range(1, self.max_steps + 1):
            step_start = time.time()

            # Get LLM response
            response = generate(messages, max_new_tokens=512, temperature=0.5)
            parsed, tool_result = self.dispatcher.dispatch(response)

            action = parsed.get("action", "unknown")
            step_time = time.time() - step_start

            step_record = {
                "step": step,
                "action": action,
                "parsed": parsed,
                "tool_result": tool_result,
                "time": step_time,
            }
            trace.append(step_record)

            # Print trace
            if action == "think":
                thought = parsed.get("thought", "")
                print(f"  Step {step} [THINK] ({step_time:.1f}s): {thought[:120]}")
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": "Continue. Use a tool or provide your answer."})

            elif action == "tool":
                tool_name = parsed.get("tool", "?")
                args = parsed.get("arguments", {})
                print(f"  Step {step} [TOOL: {tool_name}] ({step_time:.1f}s)")
                print(f"    Args: {args}")
                print(f"    Result: {tool_result}")
                messages.append({"role": "assistant", "content": response})
                messages.append({
                    "role": "user",
                    "content": f"Tool result for {tool_name}: {tool_result}\n\nUse this result to continue or provide your final answer."
                })

            elif action == "answer":
                final_answer = parsed.get("answer", "")
                print(f"  Step {step} [ANSWER] ({step_time:.1f}s): {final_answer[:120]}")
                break

        elapsed = time.time() - start_time

        if final_answer is None:
            final_answer = "[Max steps reached]"
            if trace:
                last = trace[-1]["parsed"]
                if last.get("thought"):
                    final_answer = f"[Incomplete] {last['thought'][:200]}"

        result = {
            "task": task,
            "answer": final_answer,
            "steps": len(trace),
            "trace": trace,
            "tool_calls": [t for t in trace if t["action"] == "tool"],
            "elapsed": elapsed,
        }

        print(f"\n  Final: {final_answer[:200]}")
        print(f"  ({len(trace)} steps, {len(result['tool_calls'])} tool calls, {elapsed:.1f}s)")
        print(f"{'='*60}")
        return result

print("✓ ToolAgent class defined")

### 7.1 — Running the ToolAgent on Diverse Tasks

In [ ]:
agent = ToolAgent(registry, max_steps=8)

# Task 1: Calculation
print("\n" + "▶"*15 + " TASK 1: Calculation " + "◀"*15)
r1 = agent.run("What is 347 multiplied by 829? Give me the exact result.")

In [ ]:
# Task 2: Text analysis
print("\n" + "▶"*15 + " TASK 2: Text Analysis " + "◀"*15)
r2 = agent.run(
    "Analyze this text and tell me the word count and character count: "
    "'The quick brown fox jumps over the lazy dog near the riverbank'"
)

In [ ]:
# Task 3: Multi-tool task
print("\n" + "▶"*15 + " TASK 3: Multi-Tool " + "◀"*15)
r3 = agent.run(
    "What is the definition of 'recursion'? Also, tell me how many characters "
    "are in the word 'recursion' and what it looks like reversed."
)

In [ ]:
# Task 4: Date + calculation
print("\n" + "▶"*15 + " TASK 4: Date + Calculation " + "◀"*15)
r4 = agent.run("What is today's date? And what is 365 * 24 (hours in a year)?")

In [ ]:
# Task 5: Sorting + statistics
print("\n" + "▶"*15 + " TASK 5: Sort + Statistics " + "◀"*15)
r5 = agent.run(
    "Sort these numbers: 42, 17, 83, 5, 61, 29. "
    "Also provide text statistics for the sentence: 'Artificial intelligence is transforming every industry in profound ways.'"
)

## 8. Failure Modes and Error Recovery

Tool use can fail in several ways. Understanding these failures is essential for building robust agents.

| Failure Mode | Description | Recovery |
|-------------|-------------|----------|
| **Hallucinated tool** | LLM invents a tool that doesn't exist | Error message listing valid tools |
| **Wrong parameters** | Correct tool, wrong/missing arguments | Error with expected parameter info |
| **Malformed JSON** | Output isn't valid JSON | Fallback parser extracts intent |
| **Tool execution error** | Valid call but tool crashes | Catch exception, return error string |

In [ ]:
# Demonstrate failure modes
print("="*60)
print("FAILURE MODE DEMONSTRATIONS")
print("="*60)

# 1. Hallucinated tool
print("\n--- Hallucinated Tool ---")
parsed1, result1 = dispatcher.dispatch('{"action": "tool", "tool": "web_search", "arguments": {"query": "test"}}')
print(f"  Parsed: {parsed1}")
print(f"  Result: {result1}")

# 2. Missing required parameter
print("\n--- Missing Parameter ---")
parsed2, result2 = dispatcher.dispatch('{"action": "tool", "tool": "calculator", "arguments": {}}')
print(f"  Parsed: {parsed2}")
print(f"  Result: {result2}")

# 3. Malformed JSON
print("\n--- Malformed JSON ---")
parsed3, result3 = dispatcher.dispatch('I want to use the calculator tool with expression 2+2')
print(f"  Parsed: {parsed3}")
print(f"  Result: {result3}")

# 4. Tool execution error
print("\n--- Tool Execution Error ---")
parsed4, result4 = dispatcher.dispatch('{"action": "tool", "tool": "calculator", "arguments": {"expression": "1/0"}}')
print(f"  Parsed: {parsed4}")
print(f"  Result: {result4}")

print("\n✓ All failure modes handled gracefully — no crashes!")

## 9. Tool Description Quality Experiment

Does the quality of a tool's description affect how well the LLM uses it? Let's test with three description levels for the same tool.

In [ ]:
# Create three registries with different description quality
descriptions = {
    "poor": "does stuff with numbers",
    "medium": "calculator for math",
    "excellent": "Evaluate a mathematical expression and return the precise numeric result. Supports +, -, *, /, and parentheses. Example: '(10 + 5) * 3' returns '45'."
}

tasks = [
    "What is 15 times 23?",
    "Calculate the area of a rectangle with width 7.5 and height 12.3",
    "If I have 1000 dollars and spend 347, then earn 25% of what's left, what do I have?",
]

print(f"{'Task':<65} {'Poor':<10} {'Medium':<10} {'Excellent':<10}")
print("─" * 95)

for task in tasks:
    results = {}
    for quality, desc in descriptions.items():
        test_registry = ToolRegistry()
        test_registry.register_tool(name="calculator", description=desc)(calculator)
        test_agent = ToolAgent(test_registry, max_steps=5)
        r = test_agent.run(task)
        used_tool = any(s["action"] == "tool" for s in r["trace"])
        results[quality] = "✓ tool" if used_tool else "✗ no tool"

    print(f"{task:<65} {results['poor']:<10} {results['medium']:<10} {results['excellent']:<10}")

## 10. Analyzing Tool Usage Patterns

In [ ]:
# Analyze tool call patterns across all runs
all_results = [r1, r2, r3, r4, r5]

print("Tool Usage Summary")
print("=" * 60)

tool_usage = defaultdict(int)
total_calls = 0

for r in all_results:
    for tc in r["tool_calls"]:
        tool_name = tc["parsed"].get("tool", "unknown")
        tool_usage[tool_name] += 1
        total_calls += 1

print(f"\nTotal tool calls across {len(all_results)} tasks: {total_calls}")
print(f"\nTool usage frequency:")
for tool_name, count in sorted(tool_usage.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"  {tool_name:<20} {count:>3}  {bar}")

print(f"\nAverage steps per task: {sum(r['steps'] for r in all_results) / len(all_results):.1f}")
print(f"Average tool calls per task: {total_calls / len(all_results):.1f}")
print(f"Average time per task: {sum(r['elapsed'] for r in all_results) / len(all_results):.1f}s")

## 11. Key Takeaways

### What We Built
- **ParameterSchema & Tool** — typed, self-describing tool definitions
- **ToolRegistry** — auto-extracts schemas from Python functions using `inspect`
- **ToolDispatcher** — validates and executes tool calls with error handling
- **ToolAgent** — complete agent with think/tool/answer action loop
- **8 working tools** — calculator, string_length, word_count, get_date, reverse_string, list_sort, dictionary_lookup, text_statistics

### Core Insights

1. **Tools make agents reliable.** Without tools, LLMs guess at math, dates, and counts. With tools, they delegate to deterministic code.

2. **Schema quality matters.** Better descriptions → better tool selection. Treat tool descriptions as part of your prompt engineering.

3. **Error handling is essential.** LLMs *will* hallucinate tools, send wrong parameters, and produce malformed JSON. Every tool call needs validation.

4. **The agent loop is unchanged.** We added a new action type ("tool") but the core loop from Notebook 02 is identical: perceive → reason → act → repeat.

5. **Tool dispatch is a trust boundary.** The LLM decides *what*, but code decides *how* and *whether*. Never pass raw LLM output to `eval()` or system commands in production.

### What's Next
In Notebook 04, we tackle the hardest part of tool use: **reliably parsing structured output**. We'll build multiple parsing strategies and a retry mechanism.

---

```
Notebook 03 Complete — Tool Use & Function Calling
From text generation to real-world action
```